# Structured Output
In this section, we'll look at structured output. By default, LLMs produce free-form text. Sometimes we want the LLM to produce output with more structure: include references in a certain format, or have specific sections with headers. We use structured output for that

In [1]:
from openai import OpenAI

openai_client = OpenAI()

In [2]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

In [3]:
CalendarEvent.model_json_schema()

{'properties': {'name': {'title': 'Name', 'type': 'string'},
  'date': {'title': 'Date', 'type': 'string'},
  'participants': {'items': {'type': 'string'},
   'title': 'Participants',
   'type': 'array'}},
 'required': ['name', 'date', 'participants'],
 'title': 'CalendarEvent',
 'type': 'object'}

In [ ]:
# Example of using structured output with the Responses API
response = openai_client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    text_format=CalendarEvent,
)

In [6]:
response.output[0].content[0].parsed

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [ ]:
# Example without structured output, just returning text
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ], # without structured output here (text_format=CalendarEvent) it will return any text output instead of a structured object
)
print(response.output_text) 

# If we don't specify the schema, the output can be anything. We want a reliable, predictable structure.
# So its important to use text_format

Event: Science Fair  
Participants: Alice and Bob  
Day: Friday  


# Strcutured RAG 
Now let's apply what we learned to our documentation RAG system in 02-retrieval_augmented_generation

In [18]:
# Lest rebuild it here
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

Indexed 382 chunks from 95 documents


In [19]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [20]:
import json

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

In [21]:
def llm(user_prompt, instructions=None, model="gpt-4o-mini"):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [22]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)

In [23]:
answer = rag('how do I implement LLM as a Judge?')

In [26]:
print(answer)

To implement an LLM as a judge, follow these steps based on the provided context:

1. **Requirements**: Ensure you have basic Python knowledge and an OpenAI API key to use the LLM evaluator.

2. **Installation**:
   - Install the Evidently library:
     ```python
     pip install evidently
     ```

3. **Import Required Modules**:
   ```python
   import pandas as pd
   import numpy as np
   from evidently import Dataset
   from evidently import DataDefinition
   from evidently import Report
   from evidently import BinaryClassification
   from evidently.descriptors import *
   from evidently.presets import TextEvals, ValueStats, ClassificationPreset
   from evidently.metrics import *
   from evidently.llm.templates import BinaryClassificationPromptTemplate
   ```

4. **Set Your OpenAI API Key**:
   ```python
   import os
   os.environ["OPENAI_API_KEY"] = "YOUR_KEY"
   ```

5. **Create an Evaluation Dataset**:
   - Prepare a toy Q&A dataset that includes the inputs sent to the LLM, appr

In [27]:
# Now lets make it structured output
def llm_structured(
        user_prompt,
        output_type,
        instructions=None,
        model="gpt-4o-mini",
    ):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed

In [28]:
response = llm_structured(
    instructions="Extract the event information.",
    user_prompt="Alice and Bob are going to a science fair on Friday.",
    output_type=CalendarEvent,
)

In [29]:
response

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [30]:
class RAGResponse(BaseModel):
    answer: str
    found_answer: bool

In [31]:
def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )

In [32]:
answer = rag_structured('how do i do llm evals?')

In [33]:
print(answer.answer[:100])
print(answer.found_answer)

To conduct LLM evaluations, you can follow these main steps:

1. **Install Required Packages**:
   `
True


In [34]:
answer = rag_structured('how do I install kafka on windows?')

print(answer.answer[:100])
print(answer.found_answer)

The provided documentation does not contain any information regarding the installation of Kafka on W
False


In [35]:
RAGResponse.model_json_schema()

{'properties': {'answer': {'title': 'Answer', 'type': 'string'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['answer', 'found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [36]:
# Lets chaneg the class so we can return an optional answer if we don't find one
from typing import Optional

class RAGResponse(BaseModel):
    answer: Optional[str] = None
    found_answer: bool

In [38]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [40]:
from typing import Literal
from pydantic import Field

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [41]:
RAGResponse.model_json_schema()

{'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
 'properties': {'answer': {'description': "The main answer to the user's question in markdown",
   'title': 'Answer',
   'type': 'string'},
  'found_answer': {'description': 'True if relevant information was found in the documentation',
   'title': 'Found Answer',
   'type': 'boolean'},
  'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
   'title': 'Confidence',
   'type': 'number'},
  'confidence_explanation': {'description': 'Explanation about the confidence level',
   'title': 'Confidence Explanation',
   'type': 'string'},
  'answer_type': {'description': 'The category of the answer',
   'enum': ['how-to',
    'explanation',
    'troubleshooting',
    'comparison',
    'reference'],
   'title': 'Answer Type',
   'type': 'string'},
  'followup_questions': {'description': 'S

In [42]:
answer = rag_structured('how do I evaluate llms', RAGResponse)

In [43]:
print(answer.answer[:100])
print(answer.confidence)
print(answer.confidence_explanation)
print(answer.answer_type)
print(answer.followup_questions)

To evaluate LLMs (Large Language Models), you can follow these steps based on a structured approach 
0.95
The information provided outlines a clear step-by-step process to evaluate LLMs based on the context given, detailing both setup and execution steps.
how-to
['What tools can I use for LLM evaluation?', 'How do I interpret the evaluation results?', 'Can I customize the evaluation criteria?']


In [44]:
# See the difference when we ask it about something that isn't in the docs
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

In [45]:
print(answer.answer[:100])
print(answer.confidence)
print(answer.confidence_explanation)
print(answer.answer_type)
print(answer.followup_questions)

The provided context does not include any information on installing Kafka on Windows. Therefore, I c
0.0
No relevant information on Kafka installation is present in the provided context.
reference
['Can you provide general information about Kafka?', 'What are the system requirements for Kafka?', 'Where can I find documentation on Kafka installation?']


In [ ]:
# Here now we can add some nested response types to make it even more structured and useful. 
# For example, we can add a nested type for not found answers, and another for found answers with more details. 
# This allows us to have a more complex and informative response structure.

In [46]:
from pydantic import model_validator


class AnswerNotFound(BaseModel):
    explanation: str


class AnswerResponse(BaseModel):
    """
    If answer is found, 'answer' is populated.
    If no answer is found, 'answer_not_found' is populated.
    Only one of the two fields can be set at a time. Never both or neither.
    """

    answer_not_found: Optional[AnswerNotFound] = None
    found_answer: bool
    answer: Optional[RAGResponse] = None

    @model_validator(mode="after")
    def check_consistency(self):
        if self.answer is not None and self.answer_not_found is not None:
            raise ValueError("Provide either 'answer' or 'answer_not_found', not both.")

        if self.answer is None and self.answer_not_found is None:
            raise ValueError("Provide either 'answer' or 'answer_not_found'.")

        return self

In [47]:
AnswerResponse.model_json_schema()

{'$defs': {'AnswerNotFound': {'properties': {'explanation': {'title': 'Explanation',
     'type': 'string'}},
   'required': ['explanation'],
   'title': 'AnswerNotFound',
   'type': 'object'},
  'RAGResponse': {'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
   'properties': {'answer': {'description': "The main answer to the user's question in markdown",
     'title': 'Answer',
     'type': 'string'},
    'found_answer': {'description': 'True if relevant information was found in the documentation',
     'title': 'Found Answer',
     'type': 'boolean'},
    'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
     'title': 'Confidence',
     'type': 'number'},
    'confidence_explanation': {'description': 'Explanation about the confidence level',
     'title': 'Confidence Explanation',
     'type': 'string'},
    'answer_type': 

In [48]:
answer = rag_structured('how do I install kafka on windows?', AnswerResponse)
answer

AnswerResponse(answer_not_found=AnswerNotFound(explanation='The provided context does not contain information on how to install Kafka on Windows.'), found_answer=False, answer=None)

In [50]:
answer = rag_structured('how do I run llm evals?', AnswerResponse)
answer

AnswerResponse(answer_not_found=None, found_answer=True, answer=RAGResponse(answer='To run LLM evaluations (evals), follow these steps:\n\n1. **Install the Evidently Library**:\n   ```python\n   pip install evidently[llm]\n   ```\n\n2. **Connect to Evidently Cloud**:\n   Replace the token with your actual API token:\n   ```python\n   from evidently.ui.workspace.cloud import CloudWorkspace\n   ws = CloudWorkspace(token="YOUR_API_TOKEN", url="https://app.evidently.cloud")\n   ```\n\n3. **Create a Project**:\n   ```python\n   project = ws.create_project("Your Project Name", org_id="YOUR_ORG_ID")\n   project.description = "Your project description"\n   project.save()\n   ```\n\n4. **Prepare your Dataset**:\n   Create a toy dataset with questions and their expected answers:\n   ```python\n   data = [\n       ["Question1", "Expected Answer1"],\n       ["Question2", "Expected Answer2"],\n   ]\n   columns = ["question", "target_response"]\n   ref_data = pd.DataFrame(data, columns=columns)\n   

In [51]:
from pydantic import ValidationError

try:
    AnswerResponse()
except ValidationError as e:
    print("Validation error:")
    print(e)

Validation error:
1 validation error for AnswerResponse
found_answer
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
